# 📄 Pagination Scraping: Çok Sayfalı Veri Çekme

**Gerçek dünyada veriler genelde tek sayfada değil!**

## 🎯 Bu Derste Öğrenecekleriniz:
1. **Pagination Pattern'leri** - Farklı sayfalama türleri
2. **URL Pattern Recognition** - URL desenlerini tanıma
3. **Loop Logic** - Döngü mantığı
4. **Error Handling** - Hata yönetimi
5. **Performance Optimization** - Hız optimizasyonu
6. **Data Aggregation** - Veri birleştirme
7. **Progress Tracking** - İlerleme takibi

---

## 🤔 Pagination Nedir?

**Pagination**: Büyük veri setlerini küçük sayfalara bölme sistemi

### 📊 Neden Pagination Kullanılır?
- **Performance** - Sayfa yüklenme hızı
- **User Experience** - Kullanıcı deneyimi
- **Server Load** - Sunucu yükü azaltma
- **SEO** - Arama motoru optimizasyonu

### 🔍 Yaygın Pagination Türleri:
1. **Numbered Pages** - `[1] [2] [3] [Next]`
2. **Load More Button** - `[Load More...]`
3. **Infinite Scroll** - Otomatik yükleme
4. **Previous/Next** - `[< Previous] [Next >]`

In [1]:
# Gerekli kütüphaneleri yükleyelim
import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.parse import urljoin, urlparse
import json
from collections import defaultdict
import math

# Progress bar için (isteğe bağlı)
try:
    from tqdm import tqdm
    print("✅ tqdm yüklendi - Progress bar kullanılabilir")
except ImportError:
    print("⚠️ tqdm yüklü değil - Progress bar olmayacak")
    print("   Yüklemek için: pip install tqdm")
    tqdm = None

print("\n✅ Kütüphaneler yüklendi!")
print("📄 Pagination scraping dersine hazırız!")

✅ tqdm yüklendi - Progress bar kullanılabilir

✅ Kütüphaneler yüklendi!
📄 Pagination scraping dersine hazırız!


---

## 🔍 Örnek 1: Pagination Pattern Analizi

**İlk adım: Sitenin pagination sistemini anlamak**

Pagination scraping yapmadan önce:
1. **Manuel olarak** birkaç sayfa gezin
2. **URL pattern'ini** keşfedin
3. **Navigation element'lerini** bulun
4. **Sayfa sayısını** tespit edin

In [2]:
def analyze_pagination_pattern(base_url):
    """
    Bir sitenin pagination pattern'ini analiz eder
    """
    
    print(f"🔍 PAGINATION PATTERN ANALİZİ")
    print(f"🌐 Base URL: {base_url}")
    print("=" * 60)
    
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Educational Bot; Pagination Analysis)'
    })
    
    analysis_results = {
        'pagination_container': None,
        'page_links': [],
        'next_link': None,
        'url_pattern': None
    }
    
    try:
        # İlk sayfayı al
        print("\n1️⃣ İlk sayfa analiz ediliyor...")
        response = session.get(base_url)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            print(f"   ✅ Sayfa başarıyla alındı ({len(response.text)} karakter)")
            
            # Pagination container'ı bul
            pagination_selectors = [
                '.pagination', '.pager', '.page-numbers',
                '[class*="page"]', '[class*="pagination"]'
            ]
            
            pagination_container = None
            for selector in pagination_selectors:
                container = soup.select_one(selector)
                if container:
                    pagination_container = container
                    print(f"   📄 Pagination container bulundu: {selector}")
                    analysis_results['pagination_container'] = selector
                    break
            
            if pagination_container:
                # Sayfa linklerini analiz et
                page_links = pagination_container.find_all('a')
                print(f"   🔗 Bulunan sayfa linkleri: {len(page_links)} adet")
                
                for i, link in enumerate(page_links[:5]):
                    href = link.get('href')
                    text = link.get_text(strip=True)
                    
                    if href:
                        full_url = urljoin(base_url, href)
                        analysis_results['page_links'].append({
                            'text': text,
                            'href': href,
                            'full_url': full_url
                        })
                        print(f"     {i+1}. '{text}' → {href}")
                
                # Next page linkini ara
                next_patterns = ['next', 'sonraki', '>', '»']
                for link in page_links:
                    link_text = link.get_text(strip=True).lower()
                    if any(pattern in link_text for pattern in next_patterns):
                        analysis_results['next_link'] = link.get('href')
                        print(f"   ▶️ Next page linki: {link.get('href')}")
                        break
            
            # URL pattern'ini tespit et
            print("\n2️⃣ URL pattern tespiti...")
            test_patterns = [
                f"{base_url}?page=2",
                f"{base_url}/page/2",
                f"{base_url}?p=2"
            ]
            
            for pattern in test_patterns:
                try:
                    test_response = session.get(pattern)
                    if test_response.status_code == 200 and len(test_response.text) > 1000:
                        analysis_results['url_pattern'] = pattern
                        print(f"   ✅ Çalışan pattern: {pattern}")
                        break
                    time.sleep(0.5)
                except:
                    continue
            
        else:
            print(f"   ❌ Sayfa alınamadı: {response.status_code}")
            
    except Exception as e:
        print(f"   ❌ Hata: {e}")
    
    print(f"\n📊 ANALİZ SONUÇLARI:")
    print(f"   📄 Container: {'Bulundu' if analysis_results['pagination_container'] else 'Bulunamadı'}")
    print(f"   🔗 Sayfa linkleri: {len(analysis_results['page_links'])}")
    print(f"   ▶️ Next link: {'Var' if analysis_results['next_link'] else 'Yok'}")
    print(f"   🎯 URL pattern: {'Tespit edildi' if analysis_results['url_pattern'] else 'Belirsiz'}")
    
    return analysis_results

# Test için Quotes to Scrape sitesini kullanalım
test_url = "http://quotes.toscrape.com/"
print(f"🧪 Test URL: {test_url}")
analysis = analyze_pagination_pattern(test_url)

print("\n✅ Pagination pattern analizi tamamlandı!")

🧪 Test URL: http://quotes.toscrape.com/
🔍 PAGINATION PATTERN ANALİZİ
🌐 Base URL: http://quotes.toscrape.com/

1️⃣ İlk sayfa analiz ediliyor...
   ✅ Sayfa başarıyla alındı (11021 karakter)
   📄 Pagination container bulundu: .pager
   🔗 Bulunan sayfa linkleri: 1 adet
     1. 'Next→' → /page/2/
   ▶️ Next page linki: /page/2/

2️⃣ URL pattern tespiti...
   ✅ Çalışan pattern: http://quotes.toscrape.com/?page=2

📊 ANALİZ SONUÇLARI:
   📄 Container: Bulundu
   🔗 Sayfa linkleri: 1
   ▶️ Next link: Var
   🎯 URL pattern: Tespit edildi

✅ Pagination pattern analizi tamamlandı!


---

## 🔢 Örnek 2: Sayfa Numarasıyla Scraping

**En yaygın pagination türü: URL'de sayfa numarası**

Pattern örnekleri:
- `example.com/page/1`, `example.com/page/2`
- `example.com?page=1`, `example.com?page=2`

In [3]:
def extract_quotes_data(soup):
    """
    Quotes to Scrape sitesinden veri çıkarır
    """
    quotes = soup.find_all('div', class_='quote')
    page_data = []
    
    for quote in quotes:
        text_elem = quote.find('span', class_='text')
        author_elem = quote.find('small', class_='author')
        tags_elems = quote.find_all('a', class_='tag')
        
        if text_elem and author_elem:
            quote_data = {
                'text': text_elem.get_text(strip=True),
                'author': author_elem.get_text(strip=True),
                'tags': [tag.get_text(strip=True) for tag in tags_elems]
            }
            page_data.append(quote_data)
    
    return page_data

def scrape_numbered_pagination(base_url, max_pages=5):
    """
    Sayfa numarasıyla pagination scraping yapar
    """
    
    print(f"🔢 SAYFA NUMARASI İLE SCRAPING")
    print(f"🌐 Base URL: {base_url}")
    print(f"📄 Maksimum sayfa: {max_pages}")
    print("=" * 50)
    
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Educational Bot; Pagination Scraping)'
    })
    
    all_data = []
    
    # Progress bar oluştur (eğer tqdm varsa)
    if tqdm:
        page_range = tqdm(range(1, max_pages + 1), desc="Sayfa scraping", unit="sayfa")
    else:
        page_range = range(1, max_pages + 1)
    
    for page_num in page_range:
        # URL oluştur
        page_url = f"{base_url}?page={page_num}"
        
        print(f"\n📄 Sayfa {page_num}: {page_url}")
        
        try:
            response = session.get(page_url)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                page_data = extract_quotes_data(soup)
                
                if page_data:
                    all_data.extend(page_data)
                    print(f"   ✅ {len(page_data)} veri alındı (Toplam: {len(all_data)})")
                    
                    # Progress bar güncelle (eğer tqdm varsa)
                    if tqdm and hasattr(page_range, 'set_postfix'):
                        page_range.set_postfix({
                            'Toplam': len(all_data),
                            'Bu sayfa': len(page_data)
                        })
                else:
                    print(f"   ⚠️ Bu sayfada veri bulunamadı")
                    if page_num > 1:  # İlk sayfa değilse dur
                        print(f"   🏁 Pagination sonu tespit edildi")
                        break
            
            elif response.status_code == 404:
                print(f"   🏁 Sayfa bulunamadı (404) - Pagination sonu")
                break
            
            else:
                print(f"   ❌ HTTP hatası: {response.status_code}")
        
        except Exception as e:
            print(f"   ❌ İstek hatası: {e}")
        
        # Rate limiting
        time.sleep(1)
    
    print(f"\n📊 SCRAPING TAMAMLANDI!")
    print(f"   📚 Toplam veri sayısı: {len(all_data)}")
    
    return all_data

print("✅ Sayfa numarası scraping fonksiyonları hazırlandı!")

✅ Sayfa numarası scraping fonksiyonları hazırlandı!


In [4]:
# Quotes to Scrape sitesinden 5 sayfa çekelim
print("🧪 TEST: Quotes to Scrape - İlk 5 Sayfa")
print("=" * 50)

quotes_data = scrape_numbered_pagination(
    base_url="http://quotes.toscrape.com",
    max_pages=5
)

# Sonuçları analiz edelim
if quotes_data:
    print(f"\n📊 VERİ ANALİZİ:")
    print(f"   📚 Toplam alıntı sayısı: {len(quotes_data)}")
    
    # Yazarlara göre grupla
    authors = {}
    for quote in quotes_data:
        author = quote['author']
        authors[author] = authors.get(author, 0) + 1
    
    print(f"   ✍️ Farklı yazar sayısı: {len(authors)}")
    print(f"   🏆 En çok alıntısı olan yazarlar:")
    
    # En çok alıntısı olan 5 yazarı göster
    top_authors = sorted(authors.items(), key=lambda x: x[1], reverse=True)[:5]
    for author, count in top_authors:
        print(f"      • {author}: {count} alıntı")
    
    # İlk 3 alıntıyı göster
    print(f"\n📝 İLK 3 ALINTI:")
    for i, quote in enumerate(quotes_data[:3], 1):
        print(f"\n   {i}. \"{quote['text'][:60]}...\"")
        print(f"      - {quote['author']}")
        print(f"      Tags: {', '.join(quote['tags'])}")

else:
    print("❌ Veri çekilemedi!")

🧪 TEST: Quotes to Scrape - İlk 5 Sayfa
🔢 SAYFA NUMARASI İLE SCRAPING
🌐 Base URL: http://quotes.toscrape.com
📄 Maksimum sayfa: 5


Sayfa scraping:   0%|          | 0/5 [00:00<?, ?sayfa/s]


📄 Sayfa 1: http://quotes.toscrape.com?page=1


Sayfa scraping:   0%|          | 0/5 [00:00<?, ?sayfa/s, Toplam=10, Bu sayfa=10]

   ✅ 10 veri alındı (Toplam: 10)


...


📄 Sayfa 4: http://quotes.toscrape.com?page=4
   ✅ 10 veri alındı (Toplam: 40)


Sayfa scraping:  80%|████████  | 4/5 [00:05<00:01,  1.21s/sayfa, Toplam=50, Bu sayfa=10]


📄 Sayfa 5: http://quotes.toscrape.com?page=5
   ✅ 10 veri alındı (Toplam: 50)


Sayfa scraping: 100%|██████████| 5/5 [00:06<00:00,  1.21s/sayfa, Toplam=50, Bu sayfa=10]


📊 SCRAPING TAMAMLANDI!
   📚 Toplam veri sayısı: 50

📊 VERİ ANALİZİ:
   📚 Toplam alıntı sayısı: 50
   ✍️ Farklı yazar sayısı: 8
   🏆 En çok alıntısı olan yazarlar:
      • Albert Einstein: 15 alıntı
      • J.K. Rowling: 5 alıntı
      • Jane Austen: 5 alıntı
      • Marilyn Monroe: 5 alıntı
      • André Gide: 5 alıntı

📝 İLK 3 ALINTI:

   1. "“The world as we have created it is a process of our thinkin..."
      - Albert Einstein
      Tags: change, deep-thoughts, thinking, world

   2. "“It is our choices, Harry, that show what we truly are, far ..."
      - J.K. Rowling
      Tags: abilities, choices

   3. "“There are only two ways to live your life. One is as though..."
      - Albert Einstein
      Tags: inspirational, life, live, miracle, miracles


---

## ▶️ Örnek 3: Next Link ile Scraping

**URL pattern'i belirsiz olduğunda: Next page linkini takip et**

Bu yöntem şu durumlarda kullanılır:
- URL pattern'i karmaşık
- Dinamik sayfa numaraları
- SPA (Single Page Application) siteler

In [5]:
def scrape_with_next_links(start_url, max_pages=10):
    """
    Next page linklerini takip ederek scraping yapar
    """
    
    print(f"▶️ NEXT LINK İLE SCRAPING")
    print(f"🌐 Start URL: {start_url}")
    print(f"📄 Maksimum sayfa: {max_pages}")
    print("=" * 50)
    
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Educational Bot; Next Link Scraping)'
    })
    
    all_data = []
    current_url = start_url
    page_count = 0
    visited_urls = set()  # Sonsuz döngü kontrolü
    
    while current_url and page_count < max_pages:
        # Sonsuz döngü kontrolü
        if current_url in visited_urls:
            print(f"🔄 Sonsuz döngü tespit edildi: {current_url}")
            break
        
        visited_urls.add(current_url)
        page_count += 1
        
        print(f"\n📄 Sayfa {page_count}: {current_url}")
        
        try:
            response = session.get(current_url)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Veriyi çıkar
                page_data = extract_quotes_data(soup)
                
                if page_data:
                    all_data.extend(page_data)
                    print(f"   ✅ {len(page_data)} veri alındı (Toplam: {len(all_data)})")
                else:
                    print(f"   ⚠️ Bu sayfada veri bulunamadı")
                
                # Next page linkini bul
                next_url = None
                
                # Farklı next link pattern'lerini dene
                next_selectors = [
                    'a:contains("Next")',
                    'a:contains("Sonraki")',
                    'a:contains(">")',
                    'a:contains("»")',
                    '.next a',
                    '.pager-next a'
                ]
                
                # BeautifulSoup :contains desteklemez, manuel kontrol
                links = soup.find_all('a')
                for link in links:
                    link_text = link.get_text().lower().strip()
                    href = link.get('href')
                    
                    if href and any(pattern in link_text for pattern in ['next', 'sonraki', '>', '»']):
                        next_url = urljoin(current_url, href)
                        print(f"   🔗 Next link bulundu: {link_text} → {href}")
                        break
                
                # Next URL kontrol
                if next_url and next_url != current_url:
                    current_url = next_url
                else:
                    print(f"   🏁 Next page linki bulunamadı - Pagination sonu")
                    break
            
            else:
                print(f"   ❌ HTTP hatası: {response.status_code}")
                break
        
        except Exception as e:
            print(f"   ❌ İstek hatası: {e}")
            break
        
        # Rate limiting
        time.sleep(1.5)
    
    print(f"\n📊 SCRAPING TAMAMLANDI!")
    print(f"   📄 Ziyaret edilen sayfa sayısı: {page_count}")
    print(f"   📚 Toplam veri sayısı: {len(all_data)}")
    print(f"   🔗 Benzersiz URL sayısı: {len(visited_urls)}")
    
    return all_data

print("✅ Next link scraping fonksiyonu hazırlandı!")

✅ Next link scraping fonksiyonu hazırlandı!


In [6]:
# Next link yöntemi ile scraping test edelim
print("🧪 TEST: Next Link ile Scraping")
print("=" * 50)

next_data = scrape_with_next_links(
    start_url="http://quotes.toscrape.com/",
    max_pages=4
)

# Sonuçları karşılaştır
if next_data:
    print(f"\n🔍 NEXT LİNK YÖNTEMİ SONUÇLARI:")
    print(f"   📚 Toplam alıntı: {len(next_data)}")
    
    # Unique check
    unique_texts = set(quote['text'] for quote in next_data)
    print(f"   🔄 Tekrar eden alıntı var mı: {'Evet' if len(unique_texts) != len(next_data) else 'Hayır'}")
    
    # Son 2 alıntıyı göster
    print(f"\n📝 SON 2 ALINTI:")
    for i, quote in enumerate(next_data[-2:], len(next_data)-1):
        print(f"\n   {i}. \"{quote['text'][:60]}...\"")
        print(f"      - {quote['author']}")

else:
    print("❌ Next link yöntemi ile veri çekilemedi!")

🧪 TEST: Next Link ile Scraping
▶️ NEXT LINK İLE SCRAPING
🌐 Start URL: http://quotes.toscrape.com/
📄 Maksimum sayfa: 4

📄 Sayfa 1: http://quotes.toscrape.com/
   ✅ 10 veri alındı (Toplam: 10)
   🔗 Next link bulundu: next → → /page/2/

📄 Sayfa 2: http://quotes.toscrape.com/page/2/
   ✅ 10 veri alındı (Toplam: 20)
   🔗 Next link bulundu: next → → /page/3/

📄 Sayfa 3: http://quotes.toscrape.com/page/3/
   ✅ 10 veri alındı (Toplam: 30)
   🔗 Next link bulundu: next → → /page/4/

📄 Sayfa 4: http://quotes.toscrape.com/page/4/
   ✅ 10 veri alındı (Toplam: 40)
   🔗 Next link bulundu: next → → /page/5/

📊 SCRAPING TAMAMLANDI!
   📄 Ziyaret edilen sayfa sayısı: 4
   📚 Toplam veri sayısı: 40
   🔗 Benzersiz URL sayısı: 4

🔍 NEXT LİNK YÖNTEMİ SONUÇLARI:
   📚 Toplam alıntı: 40
   🔄 Tekrar eden alıntı var mı: Hayır

📝 SON 2 ALINTI:

   39. "“I have always imagined that Paradise will be a kind of libr..."
      - Jorge Luis Borges

   40. "“It is never too late to be what you might have been.”..."
      

---

## 🛡️ Örnek 4: Error Handling ve Retry Logic

**Gerçek dünyada pagination scraping'de her türlü hata olabilir**

### 🚨 Yaygın Hatalar:
1. **Network Errors** - Bağlantı sorunları
2. **Rate Limiting** - Çok hızlı istek
3. **Page Not Found** - Sayfa bulunamadı
4. **Content Changes** - Sayfa yapısı değişti
5. **Server Overload** - Sunucu aşırı yüklü
6. **IP Blocking** - IP engellemesi

In [7]:
class RobustPaginationScraper:
    """
    Hata toleranslı pagination scraper
    """
    
    def __init__(self, base_url, retry_count=3, backoff_factor=2):
        self.base_url = base_url
        self.retry_count = retry_count
        self.backoff_factor = backoff_factor
        
        # Session oluştur
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Educational Bot; Robust Scraper)'
        })
        
        # Error tracking
        self.error_log = []
        self.retry_log = []
        
    def scrape_with_retry(self, page_num):
        """Retry mekanizması ile sayfa scraping"""
        
        page_url = f"{self.base_url}?page={page_num}"
        
        for attempt in range(self.retry_count):
            try:
                # Exponential backoff
                if attempt > 0:
                    wait_time = self.backoff_factor ** attempt
                    print(f"   ⏳ Deneme {attempt + 1}, {wait_time}s bekleniyor...")
                    time.sleep(wait_time)
                
                response = self.session.get(page_url, timeout=15)
                
                # Status kod kontrolü
                if response.status_code == 200:
                    soup = BeautifulSoup(response.text, 'html.parser')
                    data = extract_quotes_data(soup)
                    
                    if attempt > 0:
                        self.retry_log.append({
                            'page': page_num,
                            'attempt': attempt + 1,
                            'success': True
                        })
                        print(f"   ✅ Sayfa {page_num} - Deneme {attempt + 1}'de başarılı")
                    
                    return data
                
                elif response.status_code == 404:
                    # 404 hatası - Pagination sonu
                    print(f"   🏁 Sayfa {page_num} - 404 (Pagination sonu)")
                    return None
                
                elif response.status_code == 429:
                    # Rate limit
                    retry_after = response.headers.get('Retry-After', 60)
                    print(f"   ⏰ Rate limit - {retry_after}s beklenecek")
                    time.sleep(int(retry_after))
                    continue
                
                elif response.status_code >= 500:
                    # Server hatası - Retry yap
                    print(f"   🔧 Server hatası {response.status_code} - Tekrar denenecek")
                    continue
                
                else:
                    # Diğer HTTP hataları
                    error_msg = f"HTTP {response.status_code}"
                    print(f"   ❌ Sayfa {page_num} - {error_msg}")
                    
                    self.error_log.append({
                        'page': page_num,
                        'error': error_msg,
                        'attempt': attempt + 1
                    })
                    
                    if attempt == self.retry_count - 1:
                        return []
            
            except requests.exceptions.Timeout:
                print(f"   ⏰ Timeout - Sayfa {page_num}, Deneme {attempt + 1}")
                continue
            
            except requests.exceptions.ConnectionError:
                print(f"   🔌 Bağlantı hatası - Sayfa {page_num}, Deneme {attempt + 1}")
                continue
            
            except Exception as e:
                error_msg = f"Beklenmeyen hata: {str(e)}"
                print(f"   💥 {error_msg}")
                
                self.error_log.append({
                    'page': page_num,
                    'error': error_msg,
                    'attempt': attempt + 1
                })
                
                if attempt == self.retry_count - 1:
                    return []
        
        # Tüm denemeler başarısız
        print(f"   ❌ Sayfa {page_num} - Tüm denemeler başarısız")
        return []
    
    def scrape_range_robust(self, start_page, end_page):
        """Hata toleranslı range scraping"""
        
        print(f"🛡️ ROBUST PAGINATION SCRAPING")
        print(f"   📄 Sayfa aralığı: {start_page}-{end_page}")
        print(f"   🔄 Retry sayısı: {self.retry_count}")
        print(f"   ⏰ Backoff factor: {self.backoff_factor}")
        print("=" * 50)
        
        all_data = []
        successful_pages = 0
        
        for page_num in range(start_page, end_page + 1):
            print(f"\n📄 Sayfa {page_num} işleniyor...")
            
            page_data = self.scrape_with_retry(page_num)
            
            if page_data is None:
                # Pagination sonu
                break
            elif page_data:
                all_data.extend(page_data)
                successful_pages += 1
                print(f"   ✅ {len(page_data)} veri eklendi (Toplam: {len(all_data)})")
            else:
                print(f"   ⚠️ Sayfa {page_num} boş döndü")
            
            # Sayfalar arası nazik bekleme
            time.sleep(1.5)
        
        return all_data, successful_pages
    
    def get_error_report(self):
        """Hata raporunu döndürür"""
        
        report = {
            'total_errors': len(self.error_log),
            'total_retries': len(self.retry_log),
            'error_types': {},
            'retry_success_rate': 0
        }
        
        # Error type analizi
        for error in self.error_log:
            error_type = error['error'].split(':')[0]
            report['error_types'][error_type] = report['error_types'].get(error_type, 0) + 1
        
        # Retry başarı oranı
        if self.retry_log:
            successful_retries = sum(1 for r in self.retry_log if r['success'])
            report['retry_success_rate'] = (successful_retries / len(self.retry_log)) * 100
        
        return report

print("✅ Robust scraper sınıfı hazırlandı!")

✅ Robust scraper sınıfı hazırlandı!


In [8]:
# Robust scraper'ı test edelim
print("🧪 TEST: Robust Error Handling")
print("=" * 50)

# Robust scraper oluştur
robust_scraper = RobustPaginationScraper(
    base_url="http://quotes.toscrape.com",
    retry_count=3,
    backoff_factor=2
)

# 6 sayfa scrape et
robust_data, success_count = robust_scraper.scrape_range_robust(1, 6)

# Error report
error_report = robust_scraper.get_error_report()

print(f"\n📊 ROBUST SCRAPING SONUÇLARI:")
print(f"   ✅ Başarılı sayfa: {success_count}")
print(f"   📚 Toplanan veri: {len(robust_data)}")
print(f"   ❌ Toplam hata: {error_report['total_errors']}")
print(f"   🔄 Retry deneme: {error_report['total_retries']}")

if error_report['total_retries'] > 0:
    print(f"   📈 Retry başarı oranı: {error_report['retry_success_rate']:.1f}%")

if error_report['error_types']:
    print(f"\n🚨 HATA TİPLERİ:")
    for error_type, count in error_report['error_types'].items():
        print(f"   • {error_type}: {count} kez")

# Veri kalitesi kontrolü
if robust_data:
    print(f"\n🔍 VERİ KALİTE KONTROLÜ:")
    
    # Duplicate check
    texts = [item['text'] for item in robust_data]
    unique_texts = set(texts)
    duplicate_count = len(texts) - len(unique_texts)
    
    print(f"   🔄 Duplicate veri: {duplicate_count}")
    print(f"   ✨ Unique veri: {len(unique_texts)}")
    
    # Veri bütünlüğü
    valid_data = [item for item in robust_data if item['text'] and item['author']]
    invalid_count = len(robust_data) - len(valid_data)
    
    print(f"   ✅ Geçerli veri: {len(valid_data)}")
    print(f"   ❌ Geçersiz veri: {invalid_count}")
    
    data_quality = (len(valid_data) / len(robust_data)) * 100 if robust_data else 0
    print(f"   📊 Veri kalitesi: {data_quality:.1f}%")

🧪 TEST: Robust Error Handling
🛡️ ROBUST PAGINATION SCRAPING
   📄 Sayfa aralığı: 1-6
   🔄 Retry sayısı: 3
   ⏰ Backoff factor: 2

📄 Sayfa 1 işleniyor...
   ✅ 10 veri eklendi (Toplam: 10)

📄 Sayfa 2 işleniyor...
   ✅ 10 veri eklendi (Toplam: 20)

📄 Sayfa 3 işleniyor...
   ✅ 10 veri eklendi (Toplam: 30)

📄 Sayfa 4 işleniyor...
   ✅ 10 veri eklendi (Toplam: 40)

📄 Sayfa 5 işleniyor...
   ✅ 10 veri eklendi (Toplam: 50)

📄 Sayfa 6 işleniyor...
   ✅ 10 veri eklendi (Toplam: 60)

📊 ROBUST SCRAPING SONUÇLARI:
   ✅ Başarılı sayfa: 6
   📚 Toplanan veri: 60
   ❌ Toplam hata: 0
   🔄 Retry deneme: 0

🔍 VERİ KALİTE KONTROLÜ:
   🔄 Duplicate veri: 50
   ✨ Unique veri: 10
   ✅ Geçerli veri: 60
   ❌ Geçersiz veri: 0
   📊 Veri kalitesi: 100.0%


---

## 🎯 En İyi Pratikler & İpuçları

### ✅ Yapılması Gerekenler

1. **Pattern Recognition**
   ```python
   # Önce manuel olarak sayfa yapısını inceleyin
   # URL pattern'ini tespit edin
   # Pagination tipini belirleyin
   ```

2. **Rate Limiting**
   ```python
   time.sleep(1)  # Sayfalar arası bekleme
   # Çok hızlı istek göndermeyin
   ```

3. **Error Handling**
   ```python
   try:
       response = session.get(url, timeout=10)
   except requests.exceptions.RequestException:
       # Hata yönetimi
   ```

4. **Progress Tracking**
   ```python
   from tqdm import tqdm
   for page in tqdm(pages, desc="Scraping"):
       # Scraping işlemi
   ```

5. **Data Validation**
   ```python
   # Her sayfadan veri geldiğini kontrol edin
   # Duplicate veri kontrolü yapın
   # Veri kalitesini ölçün
   ```

### ❌ Yapılmaması Gerekenler

- 🚫 **Sonsuz döngü** - Maksimum sayfa limiti koyun
- 🚫 **Çok hızlı istek** - Rate limiting olmadan
- 🚫 **Error ignore** - Hataları görmezden gelme
- 🚫 **Memory overflow** - Çok büyük veri setlerini RAM'de tutma
- 🚫 **No validation** - Veri doğrulama yapmama

---

## 🎓 Özet: Pagination Scraping Mastery

Bu notebook'ta **pagination scraping'in tüm yönlerini** öğrendik:

### ✅ Teknik Beceriler
- **Pattern Recognition** - URL desenlerini tanıma
- **Loop Logic** - Sayfa döngüleri
- **Next Link Following** - Dinamik navigation
- **Error Recovery** - Hata toleransı
- **Progress Monitoring** - İlerleme takibi

### ✅ Optimizasyon Teknikleri
- **Session yeniden kullanımı**
- **Exponential Backoff** - Akıllı retry
- **Memory Management** - Büyük veri setleri
- **Data Quality Control** - Veri doğrulama

### 🚀 Gerçek Dünya Hazırlığı
Artık şunları yapabilirsiniz:
- ✅ **E-ticaret** sitelerinden ürün listelerini çekme
- ✅ **Haber** sitelerinden makale arşivleri
- ✅ **Forum** sitelerinden mesaj koleksiyonları
- ✅ **API** pagination'ını handle etme

### 💡 Unutmayın
- Pagination scraping **sabır** gerektirir
- **Rate limiting** her zaman kullanın
- **Error handling** kritik önem taşır
- **Veri kalitesi** kontrolü yapın
- **Etik kurallara** uyun

---

## 📝 Pratik Ödevler

### 🟢 Başlangıç Seviyesi

1. **Pattern Analysis**
   - `http://books.toscrape.com` sitesinin pagination pattern'ini analiz edin
   - URL yapısını çıkarın
   - Next page linklerini bulun

2. **Basic Pagination**
   - Books to Scrape sitesinden ilk 5 sayfayı scrape edin
   - Kitap ismi, fiyat ve rating bilgilerini alın
   - Progress tracking ekleyin

### 🟡 Orta Seviye

1. **Next Link Scraping**
   - Next page linklerini takip ederek scraping yapın
   - Sonsuz döngü kontrolü ekleyin
   - Duplicate veri kontrolü yapın

2. **Error Handling**
   - Retry mekanizması implement edin
   - Farklı error tiplerini handle edin
   - Error log tutun

### 🔴 İleri Seviye

1. **Production Ready Scraper**
   - Config dosyası ile ayarlanabilir scraper
   - Database'e otomatik kaydetme
   - Resume capability (kaldığı yerden devam etme)

2. **Multi-Site Pagination**
   - Farklı sitelerin pagination sistemlerini handle eden generic scraper
   - Site-specific configuration
   - Universal pattern detection

### ⚠️ ETİK KURALLAR

**Bu ödevleri yaparken:**
- ✅ Rate limiting uygulayın
- ✅ Robots.txt'ye saygı gösterin
- ✅ Server'ları aşırı yüklemeyin
- ❌ Çok agresif scraping yapmayın
- ❌ Anti-scraping önlemlerini bypass etmeye çalışmayın

---

## 🎉 Tebrikler!

**Pagination scraping konusunu başarıyla tamamladınız!**

Bu notebook'ta edindiğiniz süper güçler:
- 📄 **Multi-page data extraction**
- 🔍 **Pattern recognition & analysis**
- 🛡️ **Robust error handling**
- 📊 **Progress monitoring**
- ⚡ **Performance optimization**

### 💪 Artık Yapabilecekleriniz

- ✅ **Büyük ölçekli** veri koleksiyonları
- ✅ **E-ticaret** site scraping'i
- ✅ **Haber arşivleri** çekme
- ✅ **Forum mesajları** toplama
- ✅ **API pagination** handle etme

**Unutmayın:** Büyük veri, büyük sorumluluk getirir. Hep etik olun! 🕷️❤️